In [1]:
#!/usr/bin/env python
# coding: utf-8

import requests
import pandas as pd
import time
from datetime import datetime, timedelta, timezone
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Константы
URL = "https://api.twitterapi.io/twitter/tweet/advanced_search"
HEADERS = {"X-API-Key": ""}  # подставь свой ключ

BASE_TAG = "#oott"
SINCE_STR = "2026-06-11_00:00:00_UTC"
INITIAL_UNTIL = datetime(2026, 7, 12, 0, 0, 0, tzinfo=timezone.utc)

# Настройки устойчивости
PAUSE_BETWEEN_PAGES = 2
REQUEST_TIMEOUT = 30
MAX_RETRIES = 5
RETRY_BACKOFF = 2
SAVE_EVERY_N_PAGES = 10

# Ограничители
MAX_PAGES = 5000
MAX_FALLBACKS = 50  # сколько раз можно сдвигать until при "залипании"


def create_session():
    """Сессия с автоматическими повторами при обрывах соединения."""
    session = requests.Session()
    retry = Retry(
        total=MAX_RETRIES,
        backoff_factor=RETRY_BACKOFF,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def make_query(until_dt: datetime) -> str:
    until_str = until_dt.strftime("%Y-%m-%d_%H:%M:%S_UTC")
    return f"{BASE_TAG} since:{SINCE_STR} until:{until_str}"


def parse_created_at(tweet: dict):
    """Пытаемся распарсить дату твита из типичных полей."""
    raw = tweet.get("createdAt") or tweet.get("created_at") or tweet.get("date")
    if not raw:
        return None

    fmts = [
        "%a %b %d %H:%M:%S %z %Y",   # Wed Feb 01 10:22:33 +0000 2026
        "%Y-%m-%dT%H:%M:%S.%fZ",     # 2026-02-01T10:22:33.123Z
        "%Y-%m-%dT%H:%M:%SZ",        # 2026-02-01T10:22:33Z
    ]
    for fmt in fmts:
        try:
            dt = datetime.strptime(raw, fmt)
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=timezone.utc)
            return dt.astimezone(timezone.utc)
        except Exception:
            pass
    return None


def fetch_page(session, cursor: str, page: int, query: str):
    """Загружает одну страницу с повторами при сетевых ошибках."""
    querystring = {
        "queryType": "Latest",
        "query": query,
        "cursor": cursor,
    }
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(
                URL,
                headers=HEADERS,
                params=querystring,
                timeout=REQUEST_TIMEOUT,
            )
            if response.status_code != 200:
                return None, None, None, f"HTTP {response.status_code}"

            data = response.json()
            tweets = data.get("tweets", [])
            has_next = data.get("has_next_page", False)
            next_cursor = data.get("next_cursor", "")
            return tweets, has_next, next_cursor, None

        except (
            requests.exceptions.ConnectionError,
            requests.exceptions.ChunkedEncodingError,
            ConnectionResetError,
            OSError,
            requests.exceptions.Timeout,
            requests.exceptions.ReadTimeout,
        ) as e:
            last_error = e
            if attempt < MAX_RETRIES:
                wait = RETRY_BACKOFF ** attempt
                print(f"   ⚠ Попытка {attempt}/{MAX_RETRIES}: {type(e).__name__}. Повтор через {wait:.0f} сек...")
                time.sleep(wait)
            else:
                return None, None, None, last_error

    return None, None, None, last_error


def main():
    all_tweets = []
    seen_tweet_ids = set()
    seen_cursors = set()

    until_dt = INITIAL_UNTIL
    query = make_query(until_dt)
    cursor = ""
    has_next_page = True
    page = 1
    fallbacks_used = 0

    session = create_session()

    while has_next_page and page <= MAX_PAGES:
        # Если курсор повторился в текущем query-окне — fallback по времени
        if cursor in seen_cursors:
            print("   ⚠ Повтор cursor в текущем окне, делаем fallback по времени...")
            if not all_tweets:
                print("🛑 Нет твитов для fallback. Останавливаемся.")
                break

            # Сдвигаем until по самому старому уже собранному твиту
            known_dates = [parse_created_at(t) for t in all_tweets]
            known_dates = [d for d in known_dates if d is not None]
            if not known_dates:
                print("🛑 Нет валидных дат createdAt для fallback. Останавливаемся.")
                break

            until_dt = min(known_dates) - timedelta(seconds=1)
            query = make_query(until_dt)
            cursor = ""
            seen_cursors.clear()
            fallbacks_used += 1
            if fallbacks_used > MAX_FALLBACKS:
                print("🛑 Достигнут лимит fallback. Останавливаемся.")
                break

            print(f"   ↪ Новый until: {until_dt.strftime('%Y-%m-%d %H:%M:%S UTC')}")
            time.sleep(PAUSE_BETWEEN_PAGES)
            continue

        seen_cursors.add(cursor)

        print(f"📄 Страница {page}...", end=" ")
        tweets, has_next_page, next_cursor, err = fetch_page(session, cursor, page, query)

        if err is not None:
            print(f"\n❌ Ошибка после всех попыток: {err}")
            break

        # Дедуп по id
        new_count = 0
        for t in tweets:
            tid = t.get("id")
            if tid is None:
                # если вдруг id отсутствует, добавим как есть
                all_tweets.append(t)
                new_count += 1
                continue

            tid = str(tid)
            if tid not in seen_tweet_ids:
                seen_tweet_ids.add(tid)
                all_tweets.append(t)
                new_count += 1

        print(
            f"получено {len(tweets)} твитов, новых {new_count} "
            f"(всего уникальных {len(all_tweets)})"
        )

        if not has_next_page:
            break

        # Если next_cursor сломан — fallback
        if not next_cursor or next_cursor == cursor:
            print("   ⚠ Пустой/повторный next_cursor, fallback по времени...")
            page_oldest = [parse_created_at(t) for t in tweets]
            page_oldest = [d for d in page_oldest if d is not None]

            if page_oldest:
                until_dt = min(page_oldest) - timedelta(seconds=1)
            else:
                # если в странице нет даты, fallback от общего минимума
                known_dates = [parse_created_at(t) for t in all_tweets]
                known_dates = [d for d in known_dates if d is not None]
                if not known_dates:
                    print("🛑 Нет дат для fallback. Останавливаемся.")
                    break
                until_dt = min(known_dates) - timedelta(seconds=1)

            query = make_query(until_dt)
            cursor = ""
            seen_cursors.clear()
            fallbacks_used += 1
            if fallbacks_used > MAX_FALLBACKS:
                print("🛑 Достигнут лимит fallback. Останавливаемся.")
                break

            print(f"   ↪ Новый until: {until_dt.strftime('%Y-%m-%d %H:%M:%S UTC')}")
            page += 1
            time.sleep(PAUSE_BETWEEN_PAGES)
            continue

        # Дополнительный fallback: если страница полностью дубль
        if len(tweets) > 0 and new_count == 0:
            print("   ⚠ Страница без новых id, fallback по времени...")
            page_oldest = [parse_created_at(t) for t in tweets]
            page_oldest = [d for d in page_oldest if d is not None]
            if page_oldest:
                until_dt = min(page_oldest) - timedelta(seconds=1)
                query = make_query(until_dt)
                cursor = ""
                seen_cursors.clear()
                fallbacks_used += 1
                if fallbacks_used > MAX_FALLBACKS:
                    print("🛑 Достигнут лимит fallback. Останавливаемся.")
                    break
                print(f"   ↪ Новый until: {until_dt.strftime('%Y-%m-%d %H:%M:%S UTC')}")
                page += 1
                time.sleep(PAUSE_BETWEEN_PAGES)
                continue

        if SAVE_EVERY_N_PAGES and page % SAVE_EVERY_N_PAGES == 0:
            pd.json_normalize(all_tweets).to_csv(
                "twitter_backup_progress1.csv",
                index=False,
                encoding="utf-8"
            )
            print(f"   💾 Прогресс сохранён: {len(all_tweets)} уникальных твитов")

        cursor = next_cursor
        page += 1
        time.sleep(PAUSE_BETWEEN_PAGES)

    print(f"\n✅ Загружено уникальных твитов: {len(all_tweets)}")
    tweets_df = pd.json_normalize(all_tweets)
    return tweets_df


if __name__ == "__main__":
    tweets_df3 = main()

📄 Страница 1... получено 19 твитов, новых 19 (всего уникальных 19)
📄 Страница 2... получено 20 твитов, новых 20 (всего уникальных 39)
📄 Страница 3... получено 20 твитов, новых 20 (всего уникальных 59)
📄 Страница 4... получено 20 твитов, новых 20 (всего уникальных 79)
📄 Страница 5... получено 19 твитов, новых 19 (всего уникальных 98)
📄 Страница 6... получено 20 твитов, новых 20 (всего уникальных 118)
📄 Страница 7... получено 19 твитов, новых 19 (всего уникальных 137)
📄 Страница 8... получено 18 твитов, новых 18 (всего уникальных 155)
📄 Страница 9... получено 19 твитов, новых 19 (всего уникальных 174)
📄 Страница 10... получено 18 твитов, новых 18 (всего уникальных 192)
   💾 Прогресс сохранён: 192 уникальных твитов
📄 Страница 11... получено 20 твитов, новых 20 (всего уникальных 212)
📄 Страница 12... получено 20 твитов, новых 20 (всего уникальных 232)
📄 Страница 13... получено 20 твитов, новых 20 (всего уникальных 252)
📄 Страница 14... получено 19 твитов, новых 19 (всего уникальных 271)
📄 

In [3]:
tweets_df3['createdAt']

0       Sat Jul 11 23:53:46 +0000 2026
1       Sat Jul 11 23:53:29 +0000 2026
2       Sat Jul 11 23:31:18 +0000 2026
3       Sat Jul 11 23:27:34 +0000 2026
4       Sat Jul 11 22:46:23 +0000 2026
                     ...              
3368    Thu Jun 11 03:44:12 +0000 2026
3369    Thu Jun 11 03:36:28 +0000 2026
3370    Thu Jun 11 03:30:26 +0000 2026
3371    Thu Jun 11 00:58:04 +0000 2026
3372    Thu Jun 11 00:22:01 +0000 2026
Name: createdAt, Length: 3373, dtype: object

In [5]:
june_first_part = pd.read_excel("tweet_oil_june.xlsx")

In [9]:
june_july = pd.concat([tweets_df3,june_first_part], ignore_index=True)

In [10]:
june_july['createdAt']

0       Sat Jul 11 23:53:46 +0000 2026
1       Sat Jul 11 23:53:29 +0000 2026
2       Sat Jul 11 23:31:18 +0000 2026
3       Sat Jul 11 23:27:34 +0000 2026
4       Sat Jul 11 22:46:23 +0000 2026
                     ...              
5204    Mon Jun 01 00:42:25 +0000 2026
5205    Mon Jun 01 00:33:42 +0000 2026
5206    Mon Jun 01 00:28:41 +0000 2026
5207    Mon Jun 01 00:25:44 +0000 2026
5208    Mon Jun 01 00:24:11 +0000 2026
Name: createdAt, Length: 5209, dtype: object

In [ ]:
june_july.to_excel('tweet_oil_june_july.xlsx', index = False)